# Week 10: WebSockets in JavaScript

Real-time bidirectional communication between browser and server. This notebook covers both the Native WebSocket API and Socket.IO library.

**Note:** WebSocket examples require a server. Server code is provided for each section—run it with Node.js.


## Table of Contents

### Part 1: Native WebSockets
1. WebSocket Basics
2. Client Connection & Events
3. Sending & Receiving Messages
4. WebSocket Server (Node.js)
5. Binary Data
6. Reconnection Pattern

### Part 2: Socket.IO
7. Socket.IO Introduction
8. Socket.IO Server Setup
9. Socket.IO Client
10. Events & Acknowledgements
11. Rooms & Namespaces
12. Broadcasting

### Part 3: Comparison & Patterns
13. Native vs Socket.IO
14. Real-time Chat Example


---
## Part 1: Native WebSockets
---


### 1. WebSocket Basics

**Purpose:** WebSockets provide full-duplex communication channels over a single TCP connection. Unlike HTTP (request-response), WebSockets allow both client and server to send messages at any time.

**Key Concepts:**
- **Full-duplex:** Both sides can send/receive simultaneously
- **Persistent connection:** No repeated handshakes
- **Low latency:** No HTTP overhead after connection
- **Protocol:** `ws://` (unencrypted) or `wss://` (encrypted/TLS)

[MDN: WebSocket](https://developer.mozilla.org/en-US/docs/Web/API/WebSocket) | [RFC 6455](https://datatracker.ietf.org/doc/html/rfc6455)


In [ ]:
// WebSocket connection states
console.log('WebSocket States:');
console.log('CONNECTING:', WebSocket.CONNECTING); // 0
console.log('OPEN:', WebSocket.OPEN);             // 1
console.log('CLOSING:', WebSocket.CLOSING);       // 2
console.log('CLOSED:', WebSocket.CLOSED);         // 3


### 2. Client Connection & Events

**Purpose:** Establish a WebSocket connection and handle lifecycle events.

**Events:**
- `open` - Connection established
- `message` - Data received from server
- `error` - Error occurred
- `close` - Connection closed

[MDN: WebSocket Events](https://developer.mozilla.org/en-US/docs/Web/API/WebSocket#events)


In [ ]:
// Browser Client - WebSocket Connection
// Run this in browser console after starting the server

const ws = new WebSocket('ws://localhost:8080');

// Connection opened
ws.addEventListener('open', (event) => {
  console.log('✅ Connected to server');
  console.log('Ready state:', ws.readyState); // 1 = OPEN
});

// Listen for messages
ws.addEventListener('message', (event) => {
  console.log('📨 Message from server:', event.data);
});

// Connection error
ws.addEventListener('error', (event) => {
  console.error('❌ WebSocket error:', event);
});

// Connection closed
ws.addEventListener('close', (event) => {
  console.log('🔌 Connection closed');
  console.log('Code:', event.code);
  console.log('Reason:', event.reason);
  console.log('Clean close:', event.wasClean);
});


### 3. Sending & Receiving Messages

**Purpose:** Exchange text and JSON data between client and server.

[MDN: WebSocket.send()](https://developer.mozilla.org/en-US/docs/Web/API/WebSocket/send)


In [ ]:
// Sending messages (run after connection is open)

// Send plain text
ws.send('Hello Server!');

// Send JSON data
const message = {
  type: 'chat',
  user: 'Alice',
  content: 'Hi everyone!',
  timestamp: Date.now()
};
ws.send(JSON.stringify(message));

// Check buffer status
console.log('Buffered bytes:', ws.bufferedAmount);


In [ ]:
// Receiving and parsing JSON messages

ws.addEventListener('message', (event) => {
  try {
    const data = JSON.parse(event.data);
    
    switch (data.type) {
      case 'chat':
        console.log(`[${data.user}]: ${data.content}`);
        break;
      case 'notification':
        console.log('📢', data.message);
        break;
      case 'error':
        console.error('Server error:', data.message);
        break;
      default:
        console.log('Unknown message type:', data);
    }
  } catch (e) {
    // Plain text message
    console.log('Text message:', event.data);
  }
});


### 4. WebSocket Server (Node.js)

**Purpose:** Create a WebSocket server using the `ws` library.

**Setup:**
```bash
npm install ws
```

[ws npm package](https://www.npmjs.com/package/ws)


In [ ]:
// server.js - Basic WebSocket Server
// Run with: node server.js

const { WebSocketServer } = require('ws');

const wss = new WebSocketServer({ port: 8080 });

console.log('WebSocket server running on ws://localhost:8080');

wss.on('connection', (ws, req) => {
  const clientIP = req.socket.remoteAddress;
  console.log(`New client connected from ${clientIP}`);
  
  // Send welcome message
  ws.send(JSON.stringify({
    type: 'notification',
    message: 'Welcome to the server!'
  }));
  
  // Handle incoming messages
  ws.on('message', (data) => {
    console.log('Received:', data.toString());
    
    // Echo back to client
    ws.send(`Echo: ${data}`);
    
    // Broadcast to all clients
    wss.clients.forEach((client) => {
      if (client !== ws && client.readyState === 1) {
        client.send(data.toString());
      }
    });
  });
  
  // Handle client disconnect
  ws.on('close', () => {
    console.log('Client disconnected');
  });
  
  // Handle errors
  ws.on('error', (error) => {
    console.error('WebSocket error:', error);
  });
});


### 5. Binary Data

**Purpose:** Send and receive binary data (files, images, audio) over WebSockets.

[MDN: Blob](https://developer.mozilla.org/en-US/docs/Web/API/Blob) | [MDN: ArrayBuffer](https://developer.mozilla.org/en-US/docs/Web/JavaScript/Reference/Global_Objects/ArrayBuffer)


In [ ]:
// Sending binary data from client

// Set binary type (default is 'blob')
ws.binaryType = 'arraybuffer'; // or 'blob'

// Send ArrayBuffer
const buffer = new ArrayBuffer(8);
const view = new Uint8Array(buffer);
view.set([72, 101, 108, 108, 111, 33, 33, 33]); // "Hello!!!"
ws.send(buffer);

// Send Blob (file upload)
const blob = new Blob(['Hello from Blob!'], { type: 'text/plain' });
ws.send(blob);

// Receive binary data
ws.addEventListener('message', (event) => {
  if (event.data instanceof ArrayBuffer) {
    const text = new TextDecoder().decode(event.data);
    console.log('Binary (ArrayBuffer):', text);
  } else if (event.data instanceof Blob) {
    event.data.text().then(text => {
      console.log('Binary (Blob):', text);
    });
  }
});


### 6. Reconnection Pattern

**Purpose:** Automatically reconnect when the connection drops—essential for production apps.


In [ ]:
// Robust WebSocket client with auto-reconnection

class ReconnectingWebSocket {
  constructor(url, options = {}) {
    this.url = url;
    this.maxRetries = options.maxRetries || 10;
    this.retryDelay = options.retryDelay || 1000;
    this.maxDelay = options.maxDelay || 30000;
    this.retries = 0;
    this.ws = null;
    this.listeners = { open: [], message: [], close: [], error: [] };
    
    this.connect();
  }
  
  connect() {
    console.log(`Connecting to ${this.url}...`);
    this.ws = new WebSocket(this.url);
    
    this.ws.addEventListener('open', (e) => {
      console.log('✅ Connected');
      this.retries = 0;
      this.emit('open', e);
    });
    
    this.ws.addEventListener('message', (e) => {
      this.emit('message', e);
    });
    
    this.ws.addEventListener('close', (e) => {
      this.emit('close', e);
      this.reconnect();
    });
    
    this.ws.addEventListener('error', (e) => {
      this.emit('error', e);
    });
  }
  
  reconnect() {
    if (this.retries >= this.maxRetries) {
      console.error('Max retries reached. Giving up.');
      return;
    }
    
    // Exponential backoff
    const delay = Math.min(
      this.retryDelay * Math.pow(2, this.retries),
      this.maxDelay
    );
    
    console.log(`Reconnecting in ${delay}ms (attempt ${this.retries + 1})`);
    this.retries++;
    
    setTimeout(() => this.connect(), delay);
  }
  
  send(data) {
    if (this.ws.readyState === WebSocket.OPEN) {
      this.ws.send(data);
    } else {
      console.warn('WebSocket not open. Message queued.');
    }
  }
  
  on(event, callback) {
    this.listeners[event]?.push(callback);
  }
  
  emit(event, data) {
    this.listeners[event]?.forEach(cb => cb(data));
  }
  
  close() {
    this.maxRetries = 0; // Prevent reconnection
    this.ws.close();
  }
}

// Usage
const socket = new ReconnectingWebSocket('ws://localhost:8080');

socket.on('message', (e) => {
  console.log('Received:', e.data);
});

socket.send('Hello with auto-reconnect!');


---
## Part 2: Socket.IO
---


### 7. Socket.IO Introduction

**Purpose:** Socket.IO is a library that enables real-time, bidirectional communication. It builds on WebSockets but adds features like automatic reconnection, rooms, namespaces, and fallback transports.

**Key Features:**
- Automatic reconnection
- Packet buffering during disconnection
- Acknowledgements (request-response pattern)
- Rooms and namespaces
- Fallback to HTTP long-polling
- Broadcasting

**Note:** Socket.IO is NOT a WebSocket implementation. Socket.IO clients cannot connect to plain WebSocket servers and vice versa.

[Socket.IO Documentation](https://socket.io/docs/v4/)


### 8. Socket.IO Server Setup

**Setup:**
```bash
npm install socket.io
```

[Socket.IO Server API](https://socket.io/docs/v4/server-api/)


In [ ]:
// socket-server.js - Socket.IO Server
// Run with: node socket-server.js

const { Server } = require('socket.io');
const http = require('http');

// Create HTTP server
const httpServer = http.createServer();

// Create Socket.IO server
const io = new Server(httpServer, {
  cors: {
    origin: '*', // Allow all origins (configure in production!)
    methods: ['GET', 'POST']
  }
});

// Handle connections
io.on('connection', (socket) => {
  console.log(`Client connected: ${socket.id}`);
  
  // Send welcome event
  socket.emit('welcome', {
    message: 'Welcome to Socket.IO server!',
    id: socket.id
  });
  
  // Listen for custom events
  socket.on('chat message', (data) => {
    console.log('Message received:', data);
    
    // Broadcast to all clients except sender
    socket.broadcast.emit('chat message', {
      ...data,
      from: socket.id
    });
  });
  
  // Handle disconnection
  socket.on('disconnect', (reason) => {
    console.log(`Client disconnected: ${socket.id}, reason: ${reason}`);
  });
});

const PORT = 3000;
httpServer.listen(PORT, () => {
  console.log(`Socket.IO server running on http://localhost:${PORT}`);
});


In [ ]:
// Express + Socket.IO server
// npm install express socket.io

const express = require('express');
const { createServer } = require('http');
const { Server } = require('socket.io');

const app = express();
const httpServer = createServer(app);
const io = new Server(httpServer);

// Serve static files
app.use(express.static('public'));

// REST endpoint
app.get('/api/status', (req, res) => {
  res.json({
    status: 'online',
    clients: io.sockets.sockets.size
  });
});

// Socket.IO events
io.on('connection', (socket) => {
  console.log('User connected:', socket.id);
  
  socket.on('disconnect', () => {
    console.log('User disconnected:', socket.id);
  });
});

httpServer.listen(3000, () => {
  console.log('Express + Socket.IO on http://localhost:3000');
});


### 9. Socket.IO Client

**Browser Setup:**
```html
<script src="https://cdn.socket.io/4.7.4/socket.io.min.js"></script>
```

**Node.js Setup:**
```bash
npm install socket.io-client
```

[Socket.IO Client API](https://socket.io/docs/v4/client-api/)


In [ ]:
// Browser Client
// Include socket.io client library first

const socket = io('http://localhost:3000');

// Connection events
socket.on('connect', () => {
  console.log('✅ Connected with ID:', socket.id);
});

socket.on('disconnect', (reason) => {
  console.log('🔌 Disconnected:', reason);
  
  if (reason === 'io server disconnect') {
    // Server initiated disconnect, manual reconnect needed
    socket.connect();
  }
  // Otherwise, Socket.IO will auto-reconnect
});

socket.on('connect_error', (error) => {
  console.error('❌ Connection error:', error.message);
});

// Handle welcome event from server
socket.on('welcome', (data) => {
  console.log('📨 Welcome message:', data.message);
});


In [ ]:
// Node.js Client
// const { io } = require('socket.io-client');

const socket = io('http://localhost:3000', {
  reconnection: true,
  reconnectionAttempts: 5,
  reconnectionDelay: 1000,
  timeout: 10000,
  autoConnect: true
});

socket.on('connect', () => {
  console.log('Connected to server');
  
  // Send a message
  socket.emit('chat message', {
    user: 'NodeClient',
    text: 'Hello from Node.js!'
  });
});


### 10. Events & Acknowledgements

**Purpose:** Custom events allow semantic messaging. Acknowledgements enable request-response patterns.

[Socket.IO Emitting Events](https://socket.io/docs/v4/emitting-events/)


In [ ]:
// Custom events - Client

// Emit event with data
socket.emit('chat message', {
  user: 'Alice',
  text: 'Hello everyone!',
  timestamp: Date.now()
});

// Emit with acknowledgement (callback)
socket.emit('save message', { text: 'Important!' }, (response) => {
  if (response.status === 'ok') {
    console.log('Message saved with ID:', response.id);
  } else {
    console.error('Failed to save:', response.error);
  }
});

// Listen for events
socket.on('chat message', (data) => {
  console.log(`[${data.user}]: ${data.text}`);
});

socket.on('user joined', (data) => {
  console.log(`${data.user} joined the chat`);
});

socket.on('user left', (data) => {
  console.log(`${data.user} left the chat`);
});


In [ ]:
// Custom events - Server

io.on('connection', (socket) => {
  // Handle events with acknowledgement
  socket.on('save message', (data, callback) => {
    try {
      // Save to database...
      const id = Math.random().toString(36).substr(2, 9);
      
      // Send acknowledgement
      callback({ status: 'ok', id: id });
    } catch (error) {
      callback({ status: 'error', error: error.message });
    }
  });
  
  // Emit to single client
  socket.emit('private message', { text: 'Only you can see this' });
  
  // Emit to all clients except sender
  socket.broadcast.emit('user joined', { user: socket.id });
  
  // Emit to all clients including sender
  io.emit('announcement', { text: 'New user connected!' });
});


### 11. Rooms & Namespaces

**Purpose:** Organize connections into logical groups.
- **Rooms:** Subgroups within a namespace for targeted broadcasting
- **Namespaces:** Separate communication channels on the same connection

[Socket.IO Rooms](https://socket.io/docs/v4/rooms/) | [Socket.IO Namespaces](https://socket.io/docs/v4/namespaces/)


In [ ]:
// Rooms - Server

io.on('connection', (socket) => {
  // Join a room
  socket.on('join room', (roomName) => {
    socket.join(roomName);
    console.log(`${socket.id} joined room: ${roomName}`);
    
    // Notify room members
    socket.to(roomName).emit('user joined', {
      user: socket.id,
      room: roomName
    });
  });
  
  // Leave a room
  socket.on('leave room', (roomName) => {
    socket.leave(roomName);
    socket.to(roomName).emit('user left', { user: socket.id });
  });
  
  // Send message to room
  socket.on('room message', ({ room, message }) => {
    io.to(room).emit('chat message', {
      from: socket.id,
      text: message
    });
  });
  
  // Get rooms the socket is in
  console.log('Socket rooms:', socket.rooms); // Set { socket.id, 'room1', ... }
});


In [ ]:
// Rooms - Client

// Join a room
socket.emit('join room', 'general');
socket.emit('join room', 'javascript');

// Send message to room
socket.emit('room message', {
  room: 'javascript',
  message: 'Anyone here knows TypeScript?'
});

// Leave a room
socket.emit('leave room', 'general');


In [ ]:
// Namespaces - Server

const { Server } = require('socket.io');
const io = new Server(httpServer);

// Default namespace
io.on('connection', (socket) => {
  console.log('Connected to default namespace');
});

// Custom namespace for chat
const chatNamespace = io.of('/chat');
chatNamespace.on('connection', (socket) => {
  console.log('Connected to /chat namespace');
  
  socket.on('message', (data) => {
    chatNamespace.emit('message', data);
  });
});

// Custom namespace for notifications
const notifyNamespace = io.of('/notifications');
notifyNamespace.on('connection', (socket) => {
  console.log('Connected to /notifications namespace');
  
  // Only authenticated users
  socket.on('subscribe', (userId) => {
    socket.join(`user:${userId}`);
  });
});

// Send notification to specific user
function notifyUser(userId, notification) {
  notifyNamespace.to(`user:${userId}`).emit('notification', notification);
}


In [ ]:
// Namespaces - Client

// Connect to default namespace
const defaultSocket = io('http://localhost:3000');

// Connect to chat namespace
const chatSocket = io('http://localhost:3000/chat');
chatSocket.on('connect', () => {
  console.log('Connected to chat');
  chatSocket.emit('message', { text: 'Hello chat!' });
});

// Connect to notifications namespace
const notifySocket = io('http://localhost:3000/notifications');
notifySocket.on('connect', () => {
  console.log('Connected to notifications');
  notifySocket.emit('subscribe', 'user123');
});

notifySocket.on('notification', (data) => {
  console.log('🔔 Notification:', data);
});


### 12. Broadcasting

**Purpose:** Send messages to multiple clients efficiently.

[Socket.IO Broadcasting](https://socket.io/docs/v4/broadcasting-events/)


In [ ]:
// Broadcasting patterns - Server

io.on('connection', (socket) => {
  
  // 1. Send to the sender only
  socket.emit('private', 'Only you receive this');
  
  // 2. Send to everyone except sender
  socket.broadcast.emit('broadcast', 'Everyone except sender');
  
  // 3. Send to everyone including sender
  io.emit('global', 'Everyone receives this');
  
  // 4. Send to specific room (excluding sender)
  socket.to('room1').emit('room message', 'To room1 only');
  
  // 5. Send to specific room (including sender)
  io.to('room1').emit('room message', 'To room1 including sender');
  
  // 6. Send to multiple rooms
  io.to('room1').to('room2').emit('multi room', 'To rooms 1 and 2');
  
  // 7. Send to specific socket by ID
  io.to(socketId).emit('direct', 'Direct message to specific socket');
  
  // 8. Send to everyone except specific sockets
  socket.broadcast.except('room1').emit('filtered', 'Not to room1');
});


---
## Part 3: Comparison & Patterns
---


### 13. Native WebSocket vs Socket.IO

| Feature | Native WebSocket | Socket.IO |
|---------|-----------------|----------|
| **Protocol** | Standard WebSocket (RFC 6455) | Custom protocol over WebSocket/HTTP |
| **Reconnection** | Manual implementation | Automatic with backoff |
| **Fallback** | None | HTTP long-polling |
| **Events** | Only `message` event | Custom named events |
| **Acknowledgements** | Manual implementation | Built-in callbacks |
| **Rooms** | Manual implementation | Built-in support |
| **Binary** | ArrayBuffer/Blob | Automatic detection |
| **Multiplexing** | One connection per endpoint | Namespaces on same connection |
| **Bundle Size** | Native (0kb) | ~50kb minified |
| **Compatibility** | WebSocket clients only | Socket.IO clients only |

**When to use Native WebSocket:**
- Simple real-time needs
- Minimal bundle size required
- Interoperability with other WebSocket implementations
- Full control over protocol

**When to use Socket.IO:**
- Complex real-time applications
- Need rooms/namespaces
- Browser fallback important
- Rapid development priority


### 14. Real-time Chat Example

**Purpose:** Complete example showing a simple chat application with both Native WebSocket and Socket.IO implementations.


In [ ]:
// chat-server-native.js - Native WebSocket Chat Server
// npm install ws
// Run: node chat-server-native.js

const { WebSocketServer } = require('ws');

const wss = new WebSocketServer({ port: 8080 });
const clients = new Map(); // socket -> { id, username }

function broadcast(message, exclude = null) {
  const data = JSON.stringify(message);
  wss.clients.forEach((client) => {
    if (client !== exclude && client.readyState === 1) {
      client.send(data);
    }
  });
}

wss.on('connection', (ws) => {
  const userId = Math.random().toString(36).substr(2, 9);
  clients.set(ws, { id: userId, username: `User_${userId}` });
  
  // Send welcome
  ws.send(JSON.stringify({
    type: 'system',
    message: `Welcome! Your ID: ${userId}`,
    users: Array.from(clients.values()).map(c => c.username)
  }));
  
  // Notify others
  broadcast({
    type: 'system',
    message: `${clients.get(ws).username} joined`
  }, ws);
  
  ws.on('message', (data) => {
    const msg = JSON.parse(data);
    const user = clients.get(ws);
    
    switch (msg.type) {
      case 'chat':
        broadcast({
          type: 'chat',
          user: user.username,
          message: msg.message,
          timestamp: Date.now()
        });
        break;
      case 'setName':
        const oldName = user.username;
        user.username = msg.username;
        broadcast({
          type: 'system',
          message: `${oldName} is now ${msg.username}`
        });
        break;
    }
  });
  
  ws.on('close', () => {
    const user = clients.get(ws);
    broadcast({
      type: 'system',
      message: `${user.username} left`
    });
    clients.delete(ws);
  });
});

console.log('Native WebSocket chat on ws://localhost:8080');


In [ ]:
// chat-server-socketio.js - Socket.IO Chat Server
// npm install socket.io
// Run: node chat-server-socketio.js

const { Server } = require('socket.io');
const http = require('http');

const httpServer = http.createServer();
const io = new Server(httpServer, {
  cors: { origin: '*' }
});

const users = new Map(); // socketId -> username

io.on('connection', (socket) => {
  const username = `User_${socket.id.substr(0, 6)}`;
  users.set(socket.id, username);
  
  // Send welcome
  socket.emit('welcome', {
    message: `Welcome ${username}!`,
    users: Array.from(users.values())
  });
  
  // Notify others
  socket.broadcast.emit('user joined', { username });
  
  // Handle chat messages
  socket.on('chat', (message) => {
    io.emit('chat', {
      user: users.get(socket.id),
      message,
      timestamp: Date.now()
    });
  });
  
  // Handle name change
  socket.on('setName', (newName) => {
    const oldName = users.get(socket.id);
    users.set(socket.id, newName);
    io.emit('system', { message: `${oldName} is now ${newName}` });
  });
  
  // Handle rooms
  socket.on('join room', (room) => {
    socket.join(room);
    socket.to(room).emit('system', {
      message: `${users.get(socket.id)} joined ${room}`
    });
  });
  
  socket.on('room message', ({ room, message }) => {
    io.to(room).emit('chat', {
      user: users.get(socket.id),
      message,
      room,
      timestamp: Date.now()
    });
  });
  
  // Handle disconnect
  socket.on('disconnect', () => {
    const username = users.get(socket.id);
    io.emit('user left', { username });
    users.delete(socket.id);
  });
});

httpServer.listen(3000, () => {
  console.log('Socket.IO chat on http://localhost:3000');
});


#### Chat Client HTML (works with Native WebSocket server)

Save the following as `chat-client.html` and open in browser:


In [ ]:
// chat-client.html content:
/*
<!DOCTYPE html>
<html>
<head>
  <title>WebSocket Chat</title>
  <style>
    body { font-family: system-ui, sans-serif; max-width: 600px; margin: 50px auto; padding: 20px; }
    #messages { height: 300px; overflow-y: auto; border: 1px solid #ccc; padding: 10px; margin-bottom: 10px; }
    .system { color: #666; font-style: italic; }
    .chat { margin: 5px 0; }
    .user { font-weight: bold; color: #2196F3; }
    .controls { display: flex; gap: 10px; }
    input { flex: 1; padding: 10px; font-size: 16px; }
    button { padding: 10px 20px; background: #2196F3; color: white; border: none; cursor: pointer; }
    button:hover { background: #1976D2; }
  </style>
</head>
<body>
  <h1>WebSocket Chat</h1>
  <div id="messages"></div>
  <div class="controls">
    <input type="text" id="input" placeholder="Type a message..." autofocus>
    <button onclick="sendMessage()">Send</button>
  </div>
  
  <script>
    const messages = document.getElementById('messages');
    const input = document.getElementById('input');
    
    const ws = new WebSocket('ws://localhost:8080');
    
    ws.onopen = () => addMessage('Connected to chat server', 'system');
    ws.onclose = () => addMessage('Disconnected from server', 'system');
    
    ws.onmessage = (e) => {
      const data = JSON.parse(e.data);
      if (data.type === 'system') {
        addMessage(data.message, 'system');
      } else if (data.type === 'chat') {
        addMessage(`${data.user}: ${data.message}`, 'chat');
      }
    };
    
    function addMessage(text, type) {
      const div = document.createElement('div');
      div.className = type;
      div.textContent = text;
      messages.appendChild(div);
      messages.scrollTop = messages.scrollHeight;
    }
    
    function sendMessage() {
      const text = input.value.trim();
      if (!text) return;
      ws.send(JSON.stringify({ type: 'chat', message: text }));
      input.value = '';
    }
    
    input.addEventListener('keypress', (e) => {
      if (e.key === 'Enter') sendMessage();
    });
  </script>
</body>
</html>
*/


## Exercises

1. **Echo Server:** Create a WebSocket server that echoes messages back with a timestamp.

2. **Typing Indicator:** Implement a "user is typing" feature using Socket.IO.

3. **Private Messages:** Add direct messaging between users by socket ID.
